In [1]:
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from typing import List

## Data Ingestion

In [2]:
# URL text extraction
def extract_text_from_url(url:str) -> str:
    try:
        response = requests.get(url,timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text,"html.parser")

        for tag in soup(["script","style"]):
            tag.decompose()
        
        text = soup.get_text(separator=" ")
        clean_text = " ".join(text.split())
        return clean_text
        
    except Exception as e:
        print(f"Error fetching url {url}: {e}")
        return ""
    

In [3]:
# PDF text extraction
def extract_text_from_pdf(file_path: str)-> str:
    try:
        reader = PdfReader(file_path)
        text = ""

        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text.strip()
    
    except Exception as e:
        print(f"Error featching PDF {file_path}: {e}")
        return ""

In [4]:
def load_documents(urls: List[str]) -> List[dict]:
    documents = []
    for url in urls:
        text = extract_text_from_url(url)
        if text:
            documents.append({
                "source": url,
                "content": text
            })
    return documents

In [5]:
urls = [
    "https://www.geeksforgeeks.org/python/python-programming-language-tutorial/",
    "https://www.geeksforgeeks.org/python/python-variables/"
]

docs = load_documents(urls)

docs[0]['content']

# for i, doc in enumerate(docs):
#     print(f"Document {i+1} | Source: {doc['source']}")
#     print(doc["content"])

'Python Tutorial - GeeksforGeeks Courses Tutorials Practice Jobs Python Tutorial Data Types Interview Questions Examples Quizzes DSA Python Data Science NumPy Pandas Practice Django Flask Share Your Experiences Python Fundamentals Introduction Input & Output Variables Operators Keywords Data Types Conditional Statements Loops Functions Python Data Structures String List Tuples Dictionary Set Arrays Advanced Python OOP Concepts Exception Handling File Handling Python Database MongoDB MySQL Packages Modules DSA Libraries Python GUI Data Science with Python Numpy Pandas Matplotlib Seaborn StatsModel Model Building TensorFlow PyTorch Web Development with Python Flask Django Django ORM Jinja2 Templating Django Templates REST API Build API with DRF Python Practice Quiz Practice Problems Interview Q & A Python Courses Python Programming Course Data Analytics Course with AI Tech Interview 101 Course | DSA and System Design Summer SkillUp Explore Python Tutorial Last Updated : 8 May, 2026 Pytho

## Chunking and Embedding

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb
from uuid import uuid4

d:\AIPROJECT\LocalAIAgent\BasicRagSystem\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Chunking Function
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = words[start: end]
        chunks.append(" ".join(chunk))

        start += chunk_size - overlap

    return chunks

In [8]:
# Initialize Embedding Model
def load_embedding_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

embedding_model = load_embedding_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5016.99it/s]


In [9]:
#Initialize vector DB
def init_chroma():
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="rag_collection")
    return collection

collection = init_chroma()

In [10]:
results = collection.get(limit=5)
results

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [11]:
all_chunks = []
ids = []
for doc in docs:
    chunks = chunk_text(doc["content"])

    for chunk in chunks:
        all_chunks.append(chunk)
        ids.append(str(uuid4()))

embeddings = embedding_model.encode(all_chunks).tolist()
embeddings

[[-0.05800338461995125,
  -0.044006261974573135,
  0.00857365969568491,
  0.023043667897582054,
  -0.010985796339809895,
  -0.09814741462469101,
  0.008658231236040592,
  0.0540943406522274,
  -0.12724223732948303,
  0.011536958627402782,
  -0.039141543209552765,
  -0.03089110553264618,
  -0.02829848602414131,
  -0.00260133552365005,
  0.05986011400818825,
  0.05264108628034592,
  0.014614365994930267,
  -0.044168855994939804,
  -0.03481937199831009,
  -0.06547719985246658,
  -0.0017753884894773364,
  0.038404595106840134,
  0.005603516008704901,
  0.009765922091901302,
  0.004810325801372528,
  -0.025296000763773918,
  0.04061764106154442,
  0.04890799894928932,
  -0.009628521278500557,
  -0.006746154744178057,
  -0.01675860770046711,
  0.07291147112846375,
  0.0005379411741159856,
  0.0614890493452549,
  0.016926106065511703,
  0.005850998684763908,
  0.058704674243927,
  -0.10469289124011993,
  -0.06675985455513,
  -0.007897828705608845,
  -0.046312663704156876,
  -0.024241738021373

In [12]:
collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=ids
)
results = collection.get()
for i, doc in enumerate(results["documents"]):
    print(f"Chunk {i+1}: {doc[:300]}")


Chunk 1: Python Tutorial - GeeksforGeeks Courses Tutorials Practice Jobs Python Tutorial Data Types Interview Questions Examples Quizzes DSA Python Data Science NumPy Pandas Practice Django Flask Share Your Experiences Python Fundamentals Introduction Input & Output Variables Operators Keywords Data Types Co
Chunk 2: programming languages like Java. Provides libraries and frameworks such as Django and Flask for web development and tools like Pandas, TensorFlow and Scikit-learn for artificial intelligence, machine learning and data analysis. Cross-platform, works on Windows, Mac and Linux without major changes. U
Chunk 3: encapsulation to inheritance, polymorphism, abstract classes and iterators, we'll cover the essential concepts that helps you to build modular, reusable and scalable code. Python OOP Classes and Objects ‘Self’ as Default Argument Polymorphism Inheritance Abstraction Encapsulation Iterators Exception
Chunk 4: and moving to advanced ones. Scikit-learn XGBoost / LightGBM 

## RETRIEVE AND QUERY

In [13]:
def retrieve_chunks(query, k=3):
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    return results["documents"][0]

In [14]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("AGENT_KEY"))

def query_groq(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role":"system", "content": "You are a helpful AI assistant."},
            {"role":"user", "content":prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

In [15]:
def build_prompt(context_chunks, query):
    context = "\n\n".join(context_chunks)

    return f"""
    You are an AI assistant. Answer strictly based on the provided context.
    If the answer is not in the context, say "I don't know."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """


In [16]:
query = "Tell me about encapsulation."

context_chunks = retrieve_chunks(query)
prompt = build_prompt(context_chunks, query)

query_groq(prompt)

"Encapsulation is one of the essential concepts in object-oriented programming (OOP) that helps you to build modular, reusable, and scalable code. It refers to the idea of bundling data and its associated methods that operate on that data within a single unit, called a class or object. This helps to hide the implementation details of an object from the outside world and only expose the necessary information through public methods.\n\nIn Python, encapsulation is achieved by using private variables (those that start with a double underscore) and public methods. The private variables are not directly accessible from outside the class, but the public methods can be used to access and modify them.\n\nEncapsulation provides several benefits, including:\n\n1. Data Hiding: Encapsulation helps to hide the internal implementation details of an object from the outside world, making it harder for other parts of the code to access or modify the object's internal state directly.\n2. Code Reusability